**Install:** `pip install -U langchain langchain-openai langgraph`

# 🎯 10. Multi-Agent Systems

Real-world tasks often exceed what a single agent can handle. Multi-agent systems use **specialized agents** that communicate and collaborate.

In this notebook:

1. **Why multiple agents?** — specialization and scalability
2. **Agent communication** — message passing patterns
3. **Agent handoffs** — transferring control between agents
4. **Shared state** — the blackboard pattern
5. **Building a 2-agent collaborative system**

In [34]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from typing_extensions import TypedDict, Annotated
from pydantic import BaseModel, Field
from typing import List, Literal
import operator

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"


llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
print(f'Model: {LLM_MODEL}')

Model: gpt-4o-mini


## 10.1 Why Multiple Agents?

| Single Agent | Multi-Agent |
|-------------|-------------|
| One system prompt with all instructions | Specialized prompts per role |
| One large context window | Distributed context |
| Single point of failure | Resilient through redundancy |
| Limited specialization | Deep domain expertise per agent |

### Multi-Agent Patterns

```
1. SEQUENTIAL:   Agent A → Agent B → Agent C → Result                 
2. PARALLEL:     Agent A ──┐                                          
                 Agent B ──┼── Aggregator → Result                    
                 Agent C ──┘                                          
3. SUPERVISOR:   Supervisor → assigns → Workers → report → Supervisor 
4. DEBATE:       Agent A ←→ Agent B (argue until consensus)           
```

#### 1. Sequential

<img src="images/sequential-pattern.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

#### 2. Parallel

<img src="images/parallel-pattern.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

#### 3. Supervisor

<img src="images/supervisor-pattern.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

#### 4. Debate

<img src="images/debate-pattern.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

## 10.2 Sequential Multi-Agent Pipeline

The simplest multi-agent pattern: each agent processes the output of the previous one.

<img src="images/researcher-analyst-writer.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

#### 10.2.1. Simple Nodes

In [35]:
class PipelineState(TypedDict):
    messages: Annotated[list, operator.add]
    original_query: str
    research_output: str
    analysis_output: str
    final_report: str

def researcher(state: PipelineState) -> dict:
    """Agent 1: Research the topic."""
    response = llm.invoke([
        SystemMessage(content='You are a research specialist. Gather key facts and data about the topic. Be thorough but concise.'),
        HumanMessage(content=state['original_query'])
    ])
    print(f'  📚 Researcher done')
    return {'research_output': response.content, 'messages': [AIMessage(content=f'[Researcher]: {response.content}')]}

def analyst(state: PipelineState) -> dict:
    """Agent 2: Analyze the research."""
    response = llm.invoke([
        SystemMessage(content='You are an analyst. Take the research findings and identify key insights, patterns, and implications.'),
        HumanMessage(content=f'Research findings:\n{state["research_output"]}\n\nProvide analysis.')
    ])
    print(f'  🔍 Analyst done')
    return {'analysis_output': response.content, 'messages': [AIMessage(content=f'[Analyst]: {response.content}')]}

def writer(state: PipelineState) -> dict:
    """Agent 3: Write the final report."""
    response = llm.invoke([
        SystemMessage(content='You are a technical writer. Create a clear, well-structured report from the research and analysis. Use bullet points and headers.'),
        HumanMessage(content=f'Research:\n{state["research_output"]}\n\nAnalysis:\n{state["analysis_output"]}\n\nWrite a concise report.')
    ])
    print(f'  ✍️ Writer done')
    return {'final_report': response.content, 'messages': [AIMessage(content=f'[Writer]: {response.content}')]}

# Build pipeline
pipeline = StateGraph(PipelineState)
pipeline.add_node('researcher', researcher)
pipeline.add_node('analyst', analyst)
pipeline.add_node('writer', writer)

pipeline.add_edge(START, 'researcher')
pipeline.add_edge('researcher', 'analyst')
pipeline.add_edge('analyst', 'writer')
pipeline.add_edge('writer', END)

pipeline_app = pipeline.compile()
print('Sequential pipeline compiled: Researcher → Analyst → Writer')

Sequential pipeline compiled: Researcher → Analyst → Writer


In [36]:
from IPython.display import display, Markdown

result = pipeline_app.invoke({
    'messages': [],
    'original_query': 'What are the key advantages and risks of using multi-agent AI systems in production?',
    'research_output': '',
    'analysis_output': '',
    'final_report': ''
})

display(Markdown('\n### === FINAL REPORT ==='))
display(Markdown(result['final_report']))

  📚 Researcher done
  🔍 Analyst done
  ✍️ Writer done



### === FINAL REPORT ===

# Report on Multi-Agent AI Systems in Production

## Introduction
This report presents an analysis of the key advantages and risks associated with Multi-Agent AI Systems (MAS) in production environments. The findings highlight the potential benefits of MAS while also addressing the complexities and challenges that organizations may face during implementation.

## Key Advantages of Multi-Agent AI Systems

- **Decentralization**: 
  - Enhances resilience and reduces single points of failure.
  
- **Scalability**: 
  - Easily accommodates increased workloads by adding more agents.
  
- **Flexibility and Adaptability**: 
  - Agents can adjust to real-time changes, improving responsiveness.
  
- **Improved Efficiency**: 
  - Optimizes resource allocation and reduces bottlenecks.
  
- **Enhanced Collaboration**: 
  - Facilitates teamwork and innovation through agent communication.
  
- **Autonomy**: 
  - Reduces the need for constant human oversight, freeing resources for strategic tasks.
  
- **Real-time Decision Making**: 
  - Processes data and makes decisions quickly, crucial for time-sensitive environments.
  
- **Robustness**: 
  - The failure of one agent does not compromise the entire system.

## Key Risks of Multi-Agent AI Systems

- **Complexity**: 
  - Requires sophisticated algorithms and coordination, complicating implementation and maintenance.
  
- **Inter-agent Communication Issues**: 
  - Miscommunication can lead to inefficiencies and errors.
  
- **Security Vulnerabilities**: 
  - Decentralized systems may be more susceptible to hacking and data breaches.
  
- **Resource Competition**: 
  - Agents may conflict over limited resources, hindering performance.
  
- **Lack of Standardization**: 
  - Absence of protocols can lead to interoperability issues.
  
- **Ethical and Accountability Concerns**: 
  - Raises questions about decision-making and accountability in critical scenarios.
  
- **Overfitting to Specific Tasks**: 
  - Agents may struggle to adapt to unforeseen challenges.
  
- **Data Dependency**: 
  - Poor data quality can significantly impact system performance.

## Analysis of Findings

### Key Insights
- **Decentralization and Resilience**: Mitigates risks of downtime in production.
- **Scalability and Dynamic Adaptation**: Essential for responding to market fluctuations.
- **Efficiency through Distribution**: Higher productivity levels can be achieved.
- **Real-time Decision Making**: Minimizes delays and improves operational flow.
- **Collaboration and Innovation**: Fosters problem-solving capabilities.
- **Autonomy and Human Resource Optimization**: Frees human resources for strategic initiatives.

### Patterns Identified
- **Trade-off Between Complexity and Autonomy**: Balancing autonomy with oversight is crucial.
- **Interdependence of Communication and Performance**: Effective communication is vital for efficiency.
- **Security and Ethical Considerations**: Prioritizing security and ethical guidelines is necessary.
- **Data Quality as a Performance Driver**: High-quality data is essential for optimal performance.

### Implications for Implementation
- **Strategic Planning for Complexity**: Develop strategies to manage system complexity.
- **Establishing Communication Protocols**: Implement standardized protocols for agent interaction.
- **Robust Security Frameworks**: Invest in cybersecurity measures to protect against threats.
- **Ethical Framework Development**: Create guidelines for responsible agent behavior.
- **Focus on Data Governance**: Ensure high-quality data is available for decision-making.

## Conclusion
Multi-Agent AI Systems present significant opportunities for enhancing production efficiency and adaptability. However, their successful implementation requires careful consideration of associated risks and complexities. Organizations must adopt a strategic approach, focusing on communication, security, ethical considerations, and data quality to maximize benefits while mitigating potential downsides.

#### 10.2.2 Lanchain Agents


<img src="images/lanchain-agents.png" width="80%" style="border-radius:10px;margin:12px 0;"/>


**Πλήρης Απομόνωση (Strict Encapsulation)**

Σε αυτή την υλοποίηση, το κεντρικό **PipelineState** δεν κρατάει ιστορικό μηνυμάτων. Το κάθε **create_agent** λειτουργεί ως ένα κλειστό κουτί:

- Ο **researcher_agent** μπορεί να κάνει 5-6 γύρους εσωτερικής σκέψης, να καλέσει το tool **search** πολλές φορές, και να έχει μια μεγάλη εσωτερική λίστα από μηνύματα.

- Η συνάρτησή σου **researcher(state)** τρέχει τον agent, αλλά στο τέλος "τραβάει" μόνο το τελευταίο μήνυμα (**out['messages'][-1].content**) και το σώζει ως **research_output**.

In [37]:
# Κάθε στάδιο του pipeline γίνεται create_agent. Έτσι
# κάθε ρόλος είναι αυτόνομος agent που μπορεί να καλέσει tools.
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

@tool
def search(query: str) -> str:
    """Simulated web search."""
    return f'[results for: {query}]'

researcher_agent = create_agent(
    model=llm, tools=[search],
    system_prompt='You are a research specialist. Gather facts using search().',
)
analyst_agent = create_agent(
    model=llm, tools=[],
    system_prompt='You are a senior analyst. Identify themes and implications.',
)
writer_agent = create_agent(
    model=llm, tools=[],
    system_prompt='You are a technical writer. Produce a 2-paragraph executive report.',
)

class PipelineState(TypedDict):
    original_query: str
    research_output: str
    analysis_output: str
    final_report: str

class PipelineInput(TypedDict):
    original_query: str

class PipelineOutput(TypedDict):
    final_report: str

def researcher(state):
    out = researcher_agent.invoke({'messages': [HumanMessage(content=state['original_query'])]})
    return {'research_output': out['messages'][-1].content}

def analyst(state):
    out = analyst_agent.invoke({'messages': [HumanMessage(content=state['research_output'])]})
    return {'analysis_output': out['messages'][-1].content}

def writer(state):
    text = f'Research:\n{state["research_output"]}\n\nAnalysis:\n{state["analysis_output"]}'
    out = writer_agent.invoke({'messages': [HumanMessage(content=text)]})
    return {'final_report': out['messages'][-1].content}

g = StateGraph(PipelineState,
    input_schema=PipelineInput,
    output_schema=PipelineOutput
    )
g.add_node('researcher', researcher)
g.add_node('analyst', analyst)
g.add_node('writer', writer)
g.add_edge(START, 'researcher')
g.add_edge('researcher', 'analyst')
g.add_edge('analyst', 'writer')
g.add_edge('writer', END)
pipeline_app = g.compile()


In [38]:
result = pipeline_app.invoke({
    # 'messages': [],
    'original_query': 'What are the key advantages and risks of using multi-agent AI systems in production?',
    # 'research_output': '',
    # 'analysis_output': '',
    # 'final_report': ''
})

display(Markdown('\n=== FINAL REPORT ==='))
display(Markdown(result['final_report']))


=== FINAL REPORT ===

**Executive Report on Multi-Agent AI Systems in Production**

The implementation of multi-agent AI systems in production environments presents a compelling opportunity for organizations to enhance operational efficiency, foster collaborative problem-solving, and improve resilience. Key advantages include scalability, which allows for the seamless addition of agents to manage increased workloads, and flexibility, enabling adaptation to dynamic conditions. Furthermore, the distributed nature of these systems promotes robust decision-making and problem-solving capabilities, as agents can work together to address complex challenges. However, the deployment of such systems is not without risks. The complexity of design and management, potential coordination challenges, and security vulnerabilities necessitate a thorough understanding of the technology. Additionally, ethical considerations surrounding autonomy and accountability must be addressed to ensure responsible use.

To capitalize on the benefits while mitigating risks, organizations should engage in strategic planning that includes workforce development, risk management frameworks, and interdisciplinary collaboration. Investing in training programs will equip employees with the necessary skills to design and manage these systems effectively. Establishing comprehensive risk management protocols will help address security and ethical concerns, while continuous monitoring and evaluation will ensure that the systems remain effective and aligned with organizational goals. By fostering a culture of ethical governance and interdisciplinary teamwork, organizations can navigate the complexities of multi-agent AI systems and leverage their full potential in production settings.

#### Some extra info

**Run Limits** με create_agent()

In [39]:
from langchain.tools import tool

@tool("research")
def research_tool(query: str) -> str:
    """
    Search for information about a topic and return concise research notes.

    Args:
        query: The research question or topic to investigate.
    """
    # Εδώ βάζεις την πραγματική υλοποίηση:
    # - Tavily
    # - SerpAPI
    # - δικό σου RAG retriever
    # - βάση δεδομένων
    # - internal documents
    return f"Research results for query: {query}"

In [40]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware

researcher_agent = create_agent(
    model="openai:gpt-4.1",
    tools=[research_tool],
    system_prompt="""
    You are the researcher.
    Use the research tool only when necessary.
    Return concise research notes for the analyst.
    """,
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="research",
            run_limit=3,
            exit_behavior="end",
        )
    ],
)

In [41]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class WorkflowState(TypedDict):
    topic: str
    research_notes: str
    analysis: str
    final_answer: str


def researcher_node(state: WorkflowState) -> dict:
    result = researcher_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"Research this topic: {state['topic']}"
            }
        ]
    })
    return {"research_notes": result["messages"][-1].content}


def analyst_node(state: WorkflowState) -> dict:
    # εδώ μπορείς να καλέσεις model με structured output
    analysis = f"Analysis based on: {state['research_notes']}"
    return {"analysis": analysis}


def writer_node(state: WorkflowState) -> dict:
    final = f"Final response based on: {state['analysis']}"
    return {"final_answer": final}


builder = StateGraph(WorkflowState)

builder.add_node("researcher", researcher_node)
builder.add_node("analyst", analyst_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "analyst")
builder.add_edge("analyst", "writer")
builder.add_edge("writer", END)

graph = builder.compile()

In [42]:
result = graph.invoke({
    "topic": "Latest AI agent architectures in production"
})

display(Markdown(result["final_answer"]))

Final response based on: Analysis based on: - Modern AI agent architectures in production frequently use modular and multi-agent systems, combining Large Language Models (LLMs) with specialized tools or APIs to solve complex tasks (e.g., OpenAI's GPT-4 Agents, Anthropic's Claude).
- Frameworks such as LangChain, AutoGen, and CrewAI enable practical applications by orchestrating interactions between LLMs, external tools, and databases, handling retrieval, reasoning, and action execution.
- Architectures often use retrieval-augmented generation (RAG), integrating vector databases (e.g., Pinecone, Weaviate) for real-time information updates and better response accuracy.
- Recent production systems implement tool-using agents, defined workflows, memory modules (short- and long-term), and robust safety mechanisms.
- Multi-modal agent architectures (combining text, vision, and audio) are being deployed, leveraging models like GPT-4o and Gemini for broader capabilities.
- Advances in autonomous and collaborative agents (such as AutoGPT and BabyAGI derivatives) show early adoption for enterprise process automation, data analysis, and user-facing AI assistants.

Let me know if you need specific details on any architecture or implementation.

## 10.3 Agent Handoffs - Dynamic Routing

In handoff patterns, a **router** agent decides which specialist to invoke:


<img src="images/tech-bill-general.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

### 10.3.1 Without create_agent

In [43]:
class HandoffState(TypedDict):
    messages: Annotated[list, operator.add]
    department: str
    response: str

def router(state: HandoffState) -> dict:
    """Route to the appropriate specialist."""
    last_msg = state['messages'][-1]
    content = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    response = llm.invoke([
        SystemMessage(content='Classify this query. Reply with exactly one word: technical, billing, or general'),
        HumanMessage(content=content)
    ])
    dept = response.content.strip().lower()
    print(f'  🔀 Router: {dept}')
    return {'department': dept}

def tech_agent(state: HandoffState) -> dict:
    last_msg = state['messages'][-1]
    content = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    r = llm.invoke([SystemMessage(content='You are a technical support expert. Provide detailed technical solutions.'), HumanMessage(content=content)])
    return {'response': r.content, 'messages': [AIMessage(content=r.content)]}

def billing_agent(state: HandoffState) -> dict:
    last_msg = state['messages'][-1]
    content = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    r = llm.invoke([SystemMessage(content='You are a billing specialist. Help with payment and subscription issues.'), HumanMessage(content=content)])
    return {'response': r.content, 'messages': [AIMessage(content=r.content)]}

def general_agent(state: HandoffState) -> dict:
    last_msg = state['messages'][-1]
    content = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    r = llm.invoke([SystemMessage(content='You are a helpful general assistant.'), HumanMessage(content=content)])
    return {'response': r.content, 'messages': [AIMessage(content=r.content)]}

def route_fn(state: HandoffState) -> str:
    dept = state.get('department', 'general')
    if 'tech' in dept: return 'tech_agent'
    if 'bill' in dept: return 'billing_agent'
    return 'general_agent'

handoff = StateGraph(HandoffState)
handoff.add_node('router', router)
handoff.add_node('tech_agent', tech_agent)
handoff.add_node('billing_agent', billing_agent)
handoff.add_node('general_agent', general_agent)

handoff.add_edge(START, 'router')
handoff.add_conditional_edges('router', route_fn)
handoff.add_edge('tech_agent', END)
handoff.add_edge('billing_agent', END)
handoff.add_edge('general_agent', END)

handoff_app = handoff.compile()

# Test with different query types
queries = ['My API returns a 500 error when I POST data', 'I was charged twice this month', 'What are your office hours?']
for q in queries:
    print(f'\nQ: {q}')
    r = handoff_app.invoke({'messages': [HumanMessage(content=q)], 'department': '', 'response': ''})
    print(f'A: {r["response"]}')


Q: My API returns a 500 error when I POST data
  🔀 Router: technical
A: A 500 Internal Server Error indicates that something has gone wrong on the server side while processing your request. Here are some steps to troubleshoot and resolve the issue:

### 1. **Check Server Logs**
   - The first step is to check the server logs for any error messages or stack traces that can provide more context about the error. Look for logs related to the API endpoint you are trying to access.
   - Common log files include:
     - Apache: `/var/log/apache2/error.log`
     - Nginx: `/var/log/nginx/error.log`
     - Application-specific logs (e.g., for Node.js, Python, etc.)

### 2. **Validate Input Data**
   - Ensure that the data you are sending in the POST request is correctly formatted and meets the API's requirements.
   - Check for:
     - Required fields that may be missing.
     - Incorrect data types (e.g., sending a string instead of an integer).
     - Invalid values (e.g., dates in the wrong 

### 10.3.2 With create_agent

In [44]:
# Specialist agents = create_agent, ο router είναι μικρό graph
# που τους καλεί ανάλογα με classification.
from typing import Literal
from typing_extensions import TypedDict, Annotated
import operator
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent

class HandoffState(TypedDict):
    messages: Annotated[list, operator.add]
    department: str
    response: str

tech_specialist = create_agent(
    model=llm, tools=[],
    system_prompt='You are technical support. Solve technical problems concisely.',
)
billing_specialist = create_agent(
    model=llm, tools=[],
    system_prompt='You are billing support. Handle invoices, charges, refunds.',
)
general_specialist = create_agent(
    model=llm, tools=[],
    system_prompt='You are general support. Help with anything else.',
)

def router(state):
    content = state['messages'][-1].content
    r = llm.invoke([
        SystemMessage(content='Classify the query. Reply with exactly one word: technical, billing, or general.'),
        HumanMessage(content=content),
    ])
    dept = r.content.strip().lower()
    if 'tech' in dept: dept = 'technical'
    elif 'bill' in dept: dept = 'billing'
    else: dept = 'general'
    return {'department': dept}

def tech_agent(state):
    out = tech_specialist.invoke({'messages': state['messages']})
    return {'response': out['messages'][-1].content}

def billing_agent(state):
    out = billing_specialist.invoke({'messages': state['messages']})
    return {'response': out['messages'][-1].content}

def general_agent(state):
    out = general_specialist.invoke({'messages': state['messages']})
    return {'response': out['messages'][-1].content}

def pick(state):
    return state['department']

g = StateGraph(HandoffState)
g.add_node('router', router)
g.add_node('technical', tech_agent)
g.add_node('billing', billing_agent)
g.add_node('general', general_agent)
g.add_edge(START, 'router')
g.add_conditional_edges('router', pick,
    {'technical': 'technical', 'billing': 'billing', 'general': 'general'})
g.add_edge('technical', END)
g.add_edge('billing', END)
g.add_edge('general', END)
handoff_app_2 = g.compile()


In [45]:
queries = ['My API returns a 500 error when I POST data', 'I was charged twice this month', 'What are your office hours?']
for q in queries:
    display(Markdown(f'\n**Q**: {q}'))
    r = handoff_app_2.invoke({'messages': [HumanMessage(content=q)], 'department': '', 'response': ''})
    display(Markdown(f'**A**: {r["response"]}'))


**Q**: My API returns a 500 error when I POST data

**A**: A 500 Internal Server Error indicates that something went wrong on the server side. Here are steps to troubleshoot:

1. **Check Server Logs**: Look at the server logs for detailed error messages that can provide insight into what went wrong.

2. **Validate Input Data**: Ensure that the data you are sending in the POST request is correctly formatted and meets the API's requirements.

3. **Inspect API Code**: If you have access to the API code, check for any unhandled exceptions or errors in the logic that processes the POST request.

4. **Test with Minimal Data**: Try sending a minimal valid payload to see if the error persists. This can help isolate the issue.

5. **Check Dependencies**: Ensure that all dependencies and services the API relies on are functioning correctly.

6. **Review API Documentation**: Confirm that you are using the correct endpoint and method, and that all required headers and parameters are included.

7. **Environment Issues**: If applicable, check if the issue is environment-specific (e.g., development vs. production).

If the problem persists after these steps, consider reaching out to the API provider for further assistance.


**Q**: I was charged twice this month

**A**: I apologize for the inconvenience. To assist you better, could you please provide me with the following details:

1. The date of the charges.
2. The amounts charged.
3. Any invoice numbers associated with the charges, if available.

Once I have this information, I can help you resolve the issue.


**Q**: What are your office hours?

**A**: I’m available 24/7, so you can reach out anytime you need assistance!

### 10.3.3 Direct Agent Nodes

In [46]:
from typing import Literal
from typing_extensions import TypedDict, Annotated

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.agents import create_agent


class HandoffState(TypedDict):
    messages: Annotated[list, add_messages]


tech_specialist = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are technical support. Solve technical problems concisely.",
)

billing_specialist = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are billing support. Handle invoices, charges, refunds.",
)

general_specialist = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are general support. Help with anything else.",
)


def route_from_start(state: HandoffState) -> Literal["technical", "billing", "general"]:
    content = state["messages"][-1].content

    r = llm.invoke([
        SystemMessage(
            content="Classify the query. Reply with exactly one word: technical, billing, or general."
        ),
        HumanMessage(content=content),
    ])

    dept = r.content.strip().lower()

    if "tech" in dept:
        return "technical"
    elif "bill" in dept:
        return "billing"
    else:
        return "general"


g = StateGraph(HandoffState)

# Εδώ συνδέεις απευθείας τους agents.
# Δεν έχεις tech_agent(), billing_agent(), general_agent().
g.add_node("technical", tech_specialist)
g.add_node("billing", billing_specialist)
g.add_node("general", general_specialist)

# Conditional entry point: ξεκινάς απευθείας από START
# και επιλέγεις ποιος agent θα τρέξει.
g.add_conditional_edges(
    START,
    route_from_start,
    {
        "technical": "technical",
        "billing": "billing",
        "general": "general",
    },
)

g.add_edge("technical", END)
g.add_edge("billing", END)
g.add_edge("general", END)

handoff_app = g.compile()

In [47]:
result = handoff_app.invoke({
    "messages": [
        HumanMessage(content="What are your support opening hours?")
    ]
})

print("Response:", result['messages'][-1].content)

Response: I’m available 24/7, so you can reach out for support anytime you need assistance!


## 10.4 Debate Pattern - Agents That Argue

Two agents with opposing viewpoints debate until they reach a consensus — this produces more balanced, well-reasoned outputs.

<img src="images/debate-pro-con.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

In [48]:
def debate(topic: str, max_rounds: int = 3) -> str:
    """Two agents debate a topic to reach a balanced conclusion."""
    pro_history = []
    con_history = []

    for round_num in range(1, max_rounds + 1):
        display(Markdown(f'\n--- Round {round_num} ---'))

        # PRO agent
        pro_context = '\n'.join(con_history[-2:]) if con_history else 'Start the debate.'
        pro_resp = llm.invoke([
            SystemMessage(content='You argue IN FAVOR. Be concise (2-3 sentences). Address counterpoints.'),
            HumanMessage(content=f'Topic: {topic}\nOpponent said: {pro_context}')
        ])
        pro_history.append(pro_resp.content)
        display(Markdown(f'  PRO: {pro_resp.content}'))

        # CON agent
        con_resp = llm.invoke([
            SystemMessage(content='You argue AGAINST. Be concise (2-3 sentences). Address counterpoints.'),
            HumanMessage(content=f'Topic: {topic}\nOpponent said: {pro_resp.content}')
        ])
        con_history.append(con_resp.content)
        display(Markdown(f'  CON: {con_resp.content}'))

    # Synthesizer
    synthesis = llm.invoke([
        SystemMessage(content='Synthesize both viewpoints into a balanced conclusion.'),
        HumanMessage(content=f'Topic: {topic}\nPRO arguments: {pro_history}\nCON arguments: {con_history}')
    ])
    return synthesis.content

conclusion = debate('Should AI agents be given the ability to autonomously execute code in production?')
display(Markdown(f'\n\n === CONCLUSION === \n{conclusion}'))


--- Round 1 ---

  PRO: AI agents should be given the ability to autonomously execute code in production because they can significantly enhance efficiency, reduce human error, and respond to issues in real-time. While concerns about security and accountability are valid, implementing robust oversight mechanisms and fail-safes can mitigate these risks, allowing organizations to leverage AI's capabilities safely. Ultimately, the benefits of increased productivity and innovation outweigh the potential drawbacks.

  CON: Allowing AI agents to autonomously execute code in production poses significant risks that cannot be fully mitigated by oversight mechanisms. The potential for catastrophic errors, security vulnerabilities, and lack of accountability remains high, as AI systems can behave unpredictably in complex environments. Moreover, reliance on AI for critical operations could lead to a loss of human expertise and oversight, ultimately undermining the very efficiency and innovation that proponents advocate.


--- Round 2 ---

  PRO: While concerns about risks and unpredictability are valid, the potential benefits of AI agents executing code autonomously in production can outweigh these risks when implemented with robust safety protocols and monitoring systems. AI can enhance efficiency, reduce human error, and quickly adapt to changing conditions, leading to improved performance and innovation. Furthermore, with proper training and oversight, AI can complement human expertise rather than replace it, allowing for a more collaborative approach to problem-solving.

  CON: While the potential benefits of AI agents executing code autonomously are appealing, the inherent unpredictability of AI systems poses significant risks that cannot be fully mitigated by safety protocols. Even with monitoring, the complexity of AI behavior can lead to unforeseen consequences that may result in catastrophic failures, especially in critical systems. Relying on AI to complement human expertise also risks diminishing accountability, as it becomes unclear who is responsible for errors or malfunctions.


--- Round 3 ---

  PRO: AI agents executing code autonomously can significantly enhance efficiency and innovation by rapidly processing vast amounts of data and executing tasks beyond human capability. While risks exist, robust safety protocols, continuous monitoring, and the ability to revert changes can mitigate these concerns effectively. Furthermore, AI can augment human expertise rather than replace it, allowing humans to focus on higher-level decision-making while AI handles routine tasks, ultimately leading to improved outcomes.

  CON: Allowing AI agents to autonomously execute code in production poses significant risks that outweigh potential benefits. Even with safety protocols, the unpredictability of AI behavior can lead to catastrophic failures, data breaches, or unintended consequences that humans may not be able to quickly address. Moreover, reliance on AI for routine tasks could erode human expertise over time, making it difficult to intervene effectively when issues arise.



 === CONCLUSION === 
The debate over whether AI agents should be granted the ability to autonomously execute code in production presents compelling arguments on both sides. Proponents highlight the potential for enhanced efficiency, reduced human error, and the ability to respond to issues in real-time, suggesting that with robust oversight mechanisms and fail-safes, the risks associated with AI can be effectively managed. They argue that AI can complement human expertise, allowing for a more collaborative approach to problem-solving and enabling humans to focus on higher-level decision-making.

Conversely, opponents caution against the significant risks that accompany such autonomy. They emphasize the unpredictability of AI systems, which can lead to catastrophic errors and security vulnerabilities that may not be fully mitigated by oversight. The concern is that reliance on AI could diminish human expertise and accountability, making it challenging to address issues when they arise.

In conclusion, while the potential benefits of allowing AI agents to autonomously execute code in production are significant, they must be carefully weighed against the inherent risks. A balanced approach may involve implementing AI in a controlled manner, where its capabilities are harnessed to enhance human decision-making while maintaining stringent oversight and accountability measures. This way, organizations can leverage the advantages of AI while safeguarding against its unpredictable nature, ensuring that human expertise remains central to critical operations.

### 10.4.1 Debate with Judge / Reflection Node

+ `langsmith`: for tracing etc

%pip install -U python-dotenv langchain langchain-openai langsmith

In [49]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import display, Markdown

# from langchain_core.tracers.langchain import wait_for_all_tracers

# wait_for_all_tracers()

_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=True)

class DebateState(TypedDict):
    topic: str
    round: int
    max_rounds: int
    pro_history: List[str]
    con_history: List[str]
    conclusion: str


def transcript(state: DebateState) -> str:
    parts = []

    for i, (pro, con) in enumerate(
        zip(state["pro_history"], state["con_history"]),
        start=1
    ):
        parts.append(f"Round {i}\nPRO: {pro}\nCON: {con}")

    return "\n\n".join(parts) if parts else "No previous arguments."


def pro_node(state: DebateState) -> DebateState:
    response = llm.invoke([
        SystemMessage(content=(
            "You argue IN FAVOR. "
            "Be concise, rigorous, and address previous counterpoints."
        )),
        HumanMessage(content=(
            f"Topic:\n{state['topic']}\n\n"
            f"Debate so far:\n{transcript(state)}"
        ))
    ])

    return {
        **state,
        "pro_history": state["pro_history"] + [response.content],
    }


def con_node(state: DebateState) -> DebateState:
    current_pro = state["pro_history"][-1]

    response = llm.invoke([
        SystemMessage(content=(
            "You argue AGAINST. "
            "Be concise, rigorous, and address the PRO argument."
        )),
        HumanMessage(content=(
            f"Topic:\n{state['topic']}\n\n"
            f"Debate so far:\n{transcript(state)}\n\n"
            f"Latest PRO argument:\n{current_pro}"
        ))
    ])

    return {
        **state,
        "con_history": state["con_history"] + [response.content],
        "round": state["round"] + 1,
    }


def synth_node(state: DebateState) -> DebateState:
    response = llm.invoke([
        SystemMessage(content=(
            "You are a neutral synthesizer. "
            "Create a balanced conclusion from both sides."
        )),
        HumanMessage(content=(
            f"Topic:\n{state['topic']}\n\n"
            f"Transcript:\n{transcript(state)}"
        ))
    ])

    return {
        **state,
        "conclusion": response.content,
    }


def should_continue(state: DebateState) -> str:
    if state["round"] < state["max_rounds"]:
        return "pro"
    return "synth"


builder = StateGraph(DebateState)

builder.add_node("pro", pro_node)
builder.add_node("con", con_node)
builder.add_node("synth", synth_node)

builder.add_edge(START, "pro")
builder.add_edge("pro", "con")

builder.add_conditional_edges(
    "con",
    should_continue,
    {
        "pro": "pro",
        "synth": "synth",
    }
)

builder.add_edge("synth", END)

debate_graph = builder.compile()

result = debate_graph.invoke({
    "topic": "Should AI agents be given the ability to autonomously execute code in production? (Answer in Greek)",
    "round": 0,
    "max_rounds": 2,
    "pro_history": [],
    "con_history": [],
    "conclusion": "",
})

display(Markdown(result["conclusion"]))

Συμπερασματικά, η συζήτηση γύρω από την αυτονομία των AI πρακτόρων να εκτελούν κώδικα σε παραγωγή αναδεικνύει σημαντικά πλεονεκτήματα και ανησυχίες. Από τη μία πλευρά, οι υποστηρικτές της αυτονομίας επισημαίνουν την αυξημένη αποδοτικότητα, τη μείωση ανθρώπινου λάθους και τη δυνατότητα συνεχούς βελτίωσης που προσφέρουν οι AI πράκτορες. Αυτές οι δυνατότητες μπορούν να οδηγήσουν σε σημαντικά οφέλη, ειδικά σε κρίσιμους τομείς όπως η υγειονομική περίθαλψη και η χρηματοδότηση.

Από την άλλη πλευρά, οι αντίπαλοι της αυτονομίας επισημαίνουν τους σοβαρούς κινδύνους που σχετίζονται με την ασφάλεια, την έλλειψη ανθρώπινης κρίσης και τις ρυθμιστικές προκλήσεις. Οι ανησυχίες αυτές είναι θεμελιώδεις, καθώς η αυτονομία των AI μπορεί να έχει καταστροφικές συνέπειες αν δεν υπάρχει επαρκής ανθρώπινη εποπτεία και ρύθμιση.

Εν τέλει, η ισορροπία μεταξύ των πλεονεκτημάτων και των κινδύνων απαιτεί προσεκτική εξέταση και στρατηγικές ρύθμισης. Η συνεργασία μεταξύ ανθρώπων και AI, καθώς και η ανάπτυξη σαφών κανονισμών, μπορεί να επιτρέψει την αξιοποίηση των δυνατοτήτων της τεχνολογίας, ενώ ταυτόχρονα θα διασφαλίσει την ασφάλεια και την ηθική υπευθυνότητα.

## 💡 Exercise 10: Build a Multi-Agent Review System

**Task**: Build a system where:
1. A **Writer** agent drafts content
2. A **Reviewer** agent evaluates it
3. A **Editor** agent incorporates feedback
4. The cycle repeats until the Reviewer approves

In [50]:
# Exercise 10: YOUR CODE HERE


## 📝 Summary

| Pattern | Architecture | Best For |
|---------|-------------|----------|
| **Sequential** | A → B → C | Pipelines with clear stages |
| **Handoff** | Router → Specialist | Customer support, classification |
| **Debate** | A ↔ B → Synthesizer | Balanced analysis, decision-making |
| **Parallel** | Fan-out → Aggregate | Independent sub-tasks |

### What's Next

In **Notebook 11: Agent Orchestration Patterns**, we scale up with supervisors, parallel execution, and hierarchical agent teams.